<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Setting-Up-pPython-Runtime-Environment" data-toc-modified-id="Setting-Up-pPython-Runtime-Environment-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Setting Up pPython Runtime Environment</a></span><ul class="toc-item"><li><span><a href="#Define-the-details-about-the-pPython-run-mode" data-toc-modified-id="Define-the-details-about-the-pPython-run-mode-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Define the details about the pPython run mode</a></span></li></ul></li><li><span><a href="#Writing-pPython-Program:-Hello-World" data-toc-modified-id="Writing-pPython-Program:-Hello-World-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Writing pPython Program: Hello World</a></span><ul class="toc-item"><li><span><a href="#Change-the-current-working-directory-to-&quot;Hello-World&quot;-tutorial" data-toc-modified-id="Change-the-current-working-directory-to-&quot;Hello-World&quot;-tutorial-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>Change the current working directory to "Hello World" tutorial</a></span></li><li><span><a href="#Code:-Hello-World-Version-1" data-toc-modified-id="Code:-Hello-World-Version-1-2.2"><span class="toc-item-num">2.2&nbsp;&nbsp;</span>Code: Hello World Version 1</a></span></li><li><span><a href="#Run-Hello-World-Version-1" data-toc-modified-id="Run-Hello-World-Version-1-2.3"><span class="toc-item-num">2.3&nbsp;&nbsp;</span>Run Hello World Version 1</a></span></li><li><span><a href="#Code:-Hello-World-Version-2" data-toc-modified-id="Code:-Hello-World-Version-2-2.4"><span class="toc-item-num">2.4&nbsp;&nbsp;</span>Code: Hello World Version 2</a></span></li><li><span><a href="#Run-Hello-World-Version-2" data-toc-modified-id="Run-Hello-World-Version-2-2.5"><span class="toc-item-num">2.5&nbsp;&nbsp;</span>Run Hello World Version 2</a></span></li><li><span><a href="#Code:-Hello-World-Version-3" data-toc-modified-id="Code:-Hello-World-Version-3-2.6"><span class="toc-item-num">2.6&nbsp;&nbsp;</span>Code: Hello World Version 3</a></span></li><li><span><a href="#Run-Hello-World-Version-3" data-toc-modified-id="Run-Hello-World-Version-3-2.7"><span class="toc-item-num">2.7&nbsp;&nbsp;</span>Run Hello World Version 3</a></span></li></ul></li></ul></div>

# Setting Up pPython Runtime Environment
<strong>This Jupyter notebook assumes that pPython configuration is already done.</strong>

In [ ]:
import platform
import os
import re
import psutil

# Set the LLSC server name for samba/nfs service
if os.path.exists('/etc/llgrid.id'):
    # On a LLSC machine
    LLSC_SERVER = '172.22.4.30'
else:
    LLSC_SERVER = 'txg-gridfs'

# Set GRID_MOUNT_PATH empty
GRID_MOUNT_PATH = ''

system_name = platform.system()
if system_name == 'Windows':
    USER = os.getenv('USERNAME')
else:
    USER = os.getenv('USER')
    
# Check LLGrid Home directory is mapped on your local system
# if not, set up your local macione for LLSC environment

# Extract mount path for LLGrid
system_name = platform.system()
if system_name == 'Windows':
    import win32net
    my_net_drive = win32net.NetUseEnum(None,0)[0]
    for drive in my_net_drive:
        # print('%s'%(drive['remote']))
        if re.search(LLSC_SERVER,drive['remote']):
            GRID_MOUNT_PATH = drive['local']
            break
else:
    USER = os.getenv('USER')
    for sdiskpart in psutil.disk_partitions(all=True):
        # print(sdiskpart)
        # print('%s'%(sdiskpart[0]))
        if re.search(LLSC_SERVER,sdiskpart[0]):
            print('Mount path: %s'%(sdiskpart[1]))
            GRID_MOUNT_PATH = sdiskpart[1]
            break

# Check variables needed to set PPYTHON_HOME environment variable for local machine 
print('User: %s'%(USER))
print('GRID_MOUNT_PATH: %s'%(GRID_MOUNT_PATH))

if os.path.exists('/etc/llgrid.id'):
    # On a LLSC machine
    HOME = os.getenv('HOME')
    PPYTHON_HOME = HOME+os.sep+'llgrid_beta'+os.sep+'pPython'+os.sep+'latest'
else:
    # On a local machine
    # LLSC HOME directory must be mapped/mounted
    if len(GRID_MOUNT_PATH) > 0 and os.path.exists(GRID_MOUNT_PATH):
        PPYTHON_HOME = GRID_MOUNT_PATH+os.sep+'llgrid_beta'+os.sep+'pPython'+os.sep+'latest'
    else:
        print('ERROR: LLSC HOME directory is not mapped/mounted on the local machine.')
        print('       Finish Section 2 first.')

print('PPYTHON_HOME: %s'%(PPYTHON_HOME))

## Define the details about the pPython run mode

<strong>
Run the following cell if pPython is called from a source installation path
</strong>

In [ ]:
import os
import sys

"""
Customization for the user runtime environment from a source installation path
"""
HOME_PATH = os.getenv('HOME')
PPYTHON_SRC_HOME = HOME_PATH + "/llgrid_beta/pPython/latest/src/pPython"
os.environ["PPYTHON_HOME"] = PPYTHON_SRC_HOME

# Add Python search path for pPython main function
PPYTHON_PATH = PPYTHON_SRC_HOME+os.sep+"src"
sys.path.append(PPYTHON_PATH)

#Update PYTHONPATH to direct to the pPython modules from the source location
PYTHONPATH=PPYTHON_SRC_HOME+os.sep+'PythonMPI'+os.sep+'src'
PYTHONPATH=PYTHONPATH+':'+PPYTHON_SRC_HOME+os.sep+'src'+os.sep+'map'
PYTHONPATH=PYTHONPATH+':'+PPYTHON_SRC_HOME+os.sep+'src'+os.sep+'dmat'
PYTHONPATH=PYTHONPATH+':'+PPYTHON_SRC_HOME+os.sep+'src'
PYTHONPATH=PYTHONPATH+':'+PPYTHON_SRC_HOME+os.sep+'sched'
PYTHONPATH=PYTHONPATH+':'+PPYTHON_SRC_HOME
os.environ["PYTHONMPI"]=PYTHONPATH
# print('PYTHONMPI=%s'%(PYTHONPATH))

print('PPYTHON_PATH: %s'%(PPYTHON_SRC_HOME))

In [ ]:
# Import PythonMPI launch funciton
from pRUN import *

# Disable HDF5 file locking (Lustre parallel filesystem on LLSC)
os.environ["HDF5_USE_FILE_LOCKING"]="FALSE"

# By default, local filesystem based messaging kernel is enabled.
# For interactive job, disable it.
os.environ['PPYTHON_LOCAL_FS'] = 'no'

# Writing pPython Program: Hello World

## Change the current working directory to "Hello World" tutorial

In [ ]:
import os
if os.path.exists('/etc/llgrid.id'):
    # On a LLSC machine
    os.chdir(HOME_PATH+os.sep+'ppython_tutorial'+os.sep+'HelloWorld')
else:
    # On a local machine
    os.chdir(GRID_MOUNT_PATH+os.sep+'ppython_tutorial'+os.sep+'HelloWorld')
    
print(os.getcwd())

## Code: Hello World Version 1
<br><strong>Print "Hello World!"</strong>

In [ ]:
src_file_v1 = 'hello_world_v1.py'
print('Read file, %s'%(src_file_v1))
print('-------------')
f = open(src_file_v1, "r")
print(f.read())
f.close()

## Run Hello World Version 1

In [ ]:
# PythonMPI script filename
py_file = 'hello_world_v1.py'

# Define number of MPI processes
n_proc = 4

# Launch pPython
pRUN( py_file, n_proc, 'grid' )

In [ ]:
# Check the PythonMPI subdirectory, which is automatically created
for item in sorted(os.listdir('PythonMPI')):
    print(item)

In [ ]:
#Check the standard output from the other processes
for i in range(3):
    fname = "PythonMPI"+os.sep+"hello_world_v1."+str(i+1)+".out"
    print('File: %s'%(fname))
    print('-------')
    fid = open(fname,"r")
    print(fid.read())
    fid.close()

## Code: Hello World Version 2
<br>Print "Hello World!"
<br><strong>Print "My process ID (Pid or Rank)"</strong>
<br><strong>Print "How many pPython processes are running (Np)"</strong>

In [ ]:
src_file_v2 = 'hello_world_v2.py'
print('Read file, %s'%(src_file_v2))
print('-------------')
f = open(src_file_v2, "r")
print(f.read())
f.close()

## Run Hello World Version 2

In [ ]:
# PythonMPI script filename
py_file = 'hello_world_v2.py'

# Define number of MPI processes
n_proc = [1, 4, 1]

# Launch pPython
pRUN( py_file, n_proc, 'grid' )

In [ ]:
# Check the PythonMPI subdirectory, which is automatically created
for item in sorted(os.listdir('PythonMPI')):
    print(item)

In [ ]:
#Check the standard output from the other processes
my_list = os.listdir('PythonMPI')
for entry in my_list:
    if re.search('p1-',entry):
        #Find the subdirectory for the first compute node 
        subdir_name = entry
        break
for i in range(3):
    fname = "PythonMPI"+os.sep+subdir_name+os.sep+"hello_world_v2."+str(i+1)+".out"
    print('File: %s'%(fname))
    print('-------')
    fid = open(fname,"r")
    print(fid.read())
    fid.close()

## Code: Hello World Version 3
<br>Print "Hello World!"
<br>Print "My process ID (Pid or Rank)"
<br>Print "How many pPython processes are running (Np)"
<br><strong>Create a distributed random array</strong>
<br><strong>Do some mathematical operations in parallel</strong>
<br><strong>Aggregate the result and display on the leader process</strong>

In [ ]:
src_file_v2 = 'hello_world_v3.py'
print('Read file, %s'%(src_file_v2))
print('-------------')
f = open(src_file_v2, "r")
print(f.read())
f.close()

## Run Hello World Version 3

In [ ]:
# PythonMPI script filename
py_file = 'hello_world_v3.py'

# Define number of MPI processes
n_proc = [1, 4, 1]

# Launch pPython
pRUN( py_file, n_proc, 'grid' )

In [ ]:
# Check the PythonMPI subdirectory, which is automatically created
for item in sorted(os.listdir('PythonMPI')):
    print(item)

In [ ]:
#Check the standard output from the other processes
my_list = os.listdir('PythonMPI')
for entry in my_list:
    if re.search('p1-',entry):
        #Find the subdirectory for the first compute node 
        subdir_name = entry
        break
for i in range(3):
    fname = "PythonMPI"+os.sep+subdir_name+os.sep+"hello_world_v3."+str(i+1)+".out"
    print('File: %s'%(fname))
    print('-------')
    fid = open(fname,"r")
    print(fid.read())
    fid.close()